## churn_data
고객이탈 예측 모델 만들기

In [27]:
import pandas as pd
import numpy as np

In [28]:
# 데이터 불러오기
df_train = pd.read_csv('gift/churn_train.csv')
df_test = pd.read_csv('gift/churn_test.csv')

In [29]:
df_train.head()

,Customer_ID,Age,Tenure_Months,Monthly_Fee,Total_Usage_GB,Support_Tickets_3M,Late_Payments_6M,Contract,AutoPay,Internet_Type,Has_Addon,NPS_Score,Region,Churn
0,22216,38,1,70096,449,1,0,Month-to-month,1,DSL,1,-57,Seoul,0
1,22583,33,11,38262,564,2,1,Month-to-month,0,Fiber,0,69,Seoul,1
2,21663,27,34,62086,362,1,0,Month-to-month,1,Fiber,1,21,Metro,1
3,23028,56,19,72615,595,0,0,1-year,0,DSL,0,16,Seoul,0
4,24344,31,62,63279,365,2,1,2-year,0,5G,0,13,Metro,0


In [30]:
df_test.head()

,Customer_ID,Age,Tenure_Months,Monthly_Fee,Total_Usage_GB,Support_Tickets_3M,Late_Payments_6M,Contract,AutoPay,Internet_Type,Has_Addon,NPS_Score,Region,Churn
0,26891,25,6,62606,450,0,1,Month-to-month,0,Fiber,0,12,Seoul,0
1,27712,29,8,57140,437,0,1,2-year,0,DSL,0,-9,Metro,0
2,25001,28,17,68910,466,2,1,Month-to-month,0,Fiber,0,24,Seoul,0
3,25854,53,23,64286,494,2,0,Month-to-month,1,Fiber,1,72,Metro,0
4,21280,63,12,62359,129,0,0,Month-to-month,1,5G,0,-41,Other,0


In [31]:
## 변수명
# churn 1(이탈) / 0(유지)
# Age : 나이
# Tenure_Months : 가입기간(개월)
# Monthly_Fee : 월 요금
# Total_Usage_GB : 최근 사용량(GB)
# Support_Tickets_3M : 최근 3개월 CS 문의 건수
# Late_Payments_6M : 최근 6개월 연체 횟수
# Contract : 계약 형태(Month-to-month / 1-year / 2-year)
# AutoPay : 자동결제(0/1)
# Internet_Type : 회선(Fiber/DSL/5G)
# Has_Addon : 부가서비스(0/1)
# NPS_Score : 만족도 점수(-100~100)
# Region : 지역(Seoul/Metro/Other)

In [32]:
# 깊은 복사
df_train_row = df_train.copy()

In [33]:
df_train.info() # 전체적인 정보 확인

<class 'pandas.DataFrame'>
RangeIndex: 6400 entries, 0 to 6399
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Customer_ID         6400 non-null   int64
 1   Age                 6400 non-null   int64
 2   Tenure_Months       6400 non-null   int64
 3   Monthly_Fee         6400 non-null   int64
 4   Total_Usage_GB      6400 non-null   int64
 5   Support_Tickets_3M  6400 non-null   int64
 6   Late_Payments_6M    6400 non-null   int64
 7   Contract            6400 non-null   str  
 8   AutoPay             6400 non-null   int64
 9   Internet_Type       6400 non-null   str  
 10  Has_Addon           6400 non-null   int64
 11  NPS_Score           6400 non-null   int64
 12  Region              6400 non-null   str  
 13  Churn               6400 non-null   int64
dtypes: int64(11), str(3)
memory usage: 700.1 KB


In [34]:
df_train.isnull().sum() # 결측치 확인

Customer_ID           0
Age                   0
Tenure_Months         0
Monthly_Fee           0
Total_Usage_GB        0
Support_Tickets_3M    0
Late_Payments_6M      0
Contract              0
AutoPay               0
Internet_Type         0
Has_Addon             0
NPS_Score             0
Region                0
Churn                 0
dtype: int64

In [35]:
# 불필요한 컬럼 제거
train = df_train.drop(['Customer_ID'], axis=1)
test = df_test.drop(['Customer_ID'], axis=1)

In [36]:
train.head()

,Age,Tenure_Months,Monthly_Fee,Total_Usage_GB,Support_Tickets_3M,Late_Payments_6M,Contract,AutoPay,Internet_Type,Has_Addon,NPS_Score,Region,Churn
0,38,1,70096,449,1,0,Month-to-month,1,DSL,1,-57,Seoul,0
1,33,11,38262,564,2,1,Month-to-month,0,Fiber,0,69,Seoul,1
2,27,34,62086,362,1,0,Month-to-month,1,Fiber,1,21,Metro,1
3,56,19,72615,595,0,0,1-year,0,DSL,0,16,Seoul,0
4,31,62,63279,365,2,1,2-year,0,5G,0,13,Metro,0


In [37]:
# 인코딩
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
train['Contract'] = encoder.fit_transform(train['Contract'])
train['Internet_Type'] = encoder.fit_transform(train['Internet_Type'])
train['Region'] = encoder.fit_transform(train['Region'])

test['Contract'] = encoder.fit_transform(test['Contract'])
test['Internet_Type'] = encoder.fit_transform(test['Internet_Type'])
test['Region'] = encoder.fit_transform(test['Region'])

In [38]:
train.head()

,Age,Tenure_Months,Monthly_Fee,Total_Usage_GB,Support_Tickets_3M,Late_Payments_6M,Contract,AutoPay,Internet_Type,Has_Addon,NPS_Score,Region,Churn
0,38,1,70096,449,1,0,2,1,1,1,-57,2,0
1,33,11,38262,564,2,1,2,0,2,0,69,2,1
2,27,34,62086,362,1,0,2,1,2,1,21,0,1
3,56,19,72615,595,0,0,0,0,1,0,16,2,0
4,31,62,63279,365,2,1,1,0,0,0,13,0,0


In [39]:
test.head()

,Age,Tenure_Months,Monthly_Fee,Total_Usage_GB,Support_Tickets_3M,Late_Payments_6M,Contract,AutoPay,Internet_Type,Has_Addon,NPS_Score,Region,Churn
0,25,6,62606,450,0,1,2,0,2,0,12,2,0
1,29,8,57140,437,0,1,1,0,1,0,-9,0,0
2,28,17,68910,466,2,1,2,0,2,0,24,2,0
3,53,23,64286,494,2,0,2,1,2,1,72,0,0
4,63,12,62359,129,0,0,2,1,0,0,-41,1,0


In [40]:
X_train = train.drop(['Churn'], axis=1)
y_train = train['Churn']

X_test = test.drop(['Churn'], axis=1)
y_test = test['Churn']

In [41]:
# test_dataset 이 있으면 스플릿 과정은 필요 없는 것인가 ?

In [42]:
# 스케일링 하기
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 모델 생성 및 학습하기 (분류모델)
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=3333)
model.fit(X_train_scaled, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [47]:
# 모델의 정확도 확인
print("학습데이터점수 :", model.score(X_train_scaled, y_train))
print("테스트데이터점수 :", model.score(X_test_scaled, y_test))

학습데이터점수 : 1.0
테스트데이터점수 : 0.78125


In [51]:
pred = model.predict(X_test_scaled)
pred

array([0, 0, 1, ..., 0, 1, 0], shape=(1600,))

In [52]:
from sklearn.metrics import accuracy_score
accuracy_score(pred, y_test)

0.78125